# Lab 2 - What fingerprints are *for*

Lab 1 asked whether the model is right. This one asks whether it is **useful**.

The setting is the one from the lecture: an analyst faces a queue of honeypot
sessions, almost every one of them a string nobody has ever seen before, because
the payload name, the drop server and the password are randomised at every run.
Reading them is not an option. LogPrécis's claim is that the *intent* underneath
is shared, and that collapsing each session to its sequence of tactics turns an
unreadable queue into a short list.

You will test that claim on **9 999 real sessions** from the CyberLab honeynet,
recorded across 20 sensors over the summer of 2019.

You will *not* run the model on them - that is the point of the exercise, not a
shortcut. The authors published LogPrécis's own word-level predictions for the
whole collection, so you analyse at the scale the paper did while your laptop
stays idle.

**What you build**

1. fingerprints, and the compression they buy
2. the analyst's queue - how many genuinely *new* things arrive per day
3. which commands are ambiguous, measured rather than asserted
4. an open characterisation of your own

*Budget: about an hour. The first cell downloads ~41 MB.*

In [ ]:
# Setup - identical in every lab notebook; works on Colab and on a local clone.
import os, subprocess, sys
from pathlib import Path

if "google.colab" in sys.modules:
    subprocess.run("pip -q install transformers torch pandas pyarrow scikit-learn matplotlib".split(), check=True)
    if not Path("Lecture-LLM_Cybersecurity").exists():
        subprocess.run("git clone -q --depth 1 https://github.com/MatteoBoffa/Lecture-LLM_Cybersecurity.git".split(), check=True)
    os.chdir("Lecture-LLM_Cybersecurity")

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "logprecis_lab.py").exists())
sys.path[:0] = [str(ROOT), str(ROOT / "labs")]

import logging, warnings                      # the Hub is chatty; we are not debugging it
warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)

import labs_lib
labs_lib.style()
print("ready -", ROOT)

In [ ]:
sessions = labs_lib.published_predictions()

print(f"{len(sessions):,} sessions | {sessions.session.nunique():,} of them unique strings")
print(f"{sessions.date.min():%Y-%m-%d} to {sessions.date.max():%Y-%m-%d} | {sessions.sensor.nunique()} sensors")
sessions[["session", "sensor", "first_timestamp"]].head(3)

Every session is its own unique string - `nunique` equals the row count. To a
`grep`, to a hash, to any exact-match indicator of compromise, these are ten
thousand different attacks.

Here is what that randomisation looks like up close:

In [ ]:
for text in sessions.session.head(3):
    print(text[:120], "\n")

Same shape, different `busybox` argument every time. Now the two columns that
matter: `sequence_words` and `sequence_predictions`, LogPrécis's own output, one
tactic per word.

In [ ]:
row = sessions.iloc[0]
list(zip(row.sequence_words, row.sequence_predictions))[:8]

## Part 1 - Fingerprints

A **fingerprint** is the session's sequence of tactics, consecutive repetitions
run-length encoded: `Discovery x 54 -- Defense Evasion x 8`. This is the paper's
own definition, and the one `logprecis_lab.fingerprint` computes in the lecture -
that function takes `(word, tactic)` pairs, and here you have a bare array of
tactics, so write the version that works on what you have.

In [ ]:
from logprecis_lab import fingerprint as lecture_fingerprint


def fingerprint_of(predictions):
    """The tactics of a session, consecutive runs encoded as 'Tactic x N'."""
    # TODO: walk `predictions`, grouping consecutive occurrences of the same tactic
    # TODO: keep each run as its tactic and how many times it repeats
    # TODO: format every run as "Tactic x N" and join them with " -- "
    raise NotImplementedError("your turn")


probe = ["Discovery", "Discovery", "Impact"]
assert fingerprint_of(probe) == "Discovery x 2 -- Impact x 1"
assert fingerprint_of(probe) == lecture_fingerprint([("w", t) for t in probe])

sessions["fingerprint"] = sessions.sequence_predictions.map(fingerprint_of)
sessions.fingerprint.head(3).tolist()

In [ ]:
n_sessions, n_fingerprints = sessions.session.nunique(), sessions.fingerprint.nunique()

print(f"{n_sessions:,} unique sessions")
print(f"{n_fingerprints:,} unique fingerprints")
print(f"-> {n_sessions / n_fingerprints:.0f}x compression")

The randomness lived in the *words*; the *intent* was shared. That is the
paper's mechanism, reproduced on a slice of its data - though not its headline
number, which is the subject of the question below.

And the distribution is kinder than the ratio suggests:

In [ ]:
counts = sessions.fingerprint.value_counts()

for fp, n in counts.head(5).items():
    print(f"{n:>5}  ({n / len(sessions):5.1%})  {fp[:66]}")
print(f"\ntop 10 fingerprints cover {counts.head(10).sum() / len(sessions):.1%} of all traffic")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

cumulative = counts.cumsum() / counts.sum()

fig, ax = plt.subplots(figsize=(7, 3.6))
ax.plot(range(1, len(cumulative) + 1), cumulative.values, color=labs_lib.SERIES[0])
ax.axhline(0.9, color=labs_lib.INK_MUTED, linewidth=1, linestyle=":")
reaches_90 = int((cumulative < 0.9).sum()) + 1
ax.annotate(f"{reaches_90} fingerprints = 90% of traffic",
            xy=(reaches_90, 0.9), xytext=(reaches_90 + 6, 0.62),
            arrowprops=dict(arrowstyle="->", color=labs_lib.INK_MUTED), color=labs_lib.INK)
ax.set(xlabel="fingerprints, most frequent first", ylabel="share of sessions covered",
       ylim=(0, 1.02), title="A handful of intents explain nearly everything")
plt.tight_layout()
plt.show()

> **Question 1.** The paper reports roughly 400 000 sessions collapsing to about
> 3 000 fingerprints - a ~130x compression - and you just measured far *less* on
> this slice. Before blaming the slice, note what kind of quantity you are
> computing: the number of distinct fingerprints is a count of *types*, and type
> counts grow sublinearly with the sample. Re-measure the ratio on the first
> 1 000, 5 000 and 9 999 sessions. Does it drift in a direction that explains the
> gap on its own? What, then, does "130x compression" actually describe - the
> attacks, or the window someone looked through?

## Part 2 - The analyst's queue

Compression is only useful if it holds *over time*. An analyst does not process a
collection, they come in on Tuesday and face what arrived on Monday.

So ask the operational question: **how many things arrive per day that we have
never seen before?**

In [ ]:
daily = sessions.groupby(sessions.date.dt.date)

volume = daily.size()
distinct = daily.fingerprint.nunique()

print(f"{len(volume)} days of traffic")
print(f"sessions per day:            median {int(volume.median()):>5}, max {volume.max():>5}")
print(f"distinct fingerprints/day:   median {int(distinct.median()):>5}, max {distinct.max():>5}")

In [ ]:
def new_per_day(frame):
    """How many fingerprints each day shows that no earlier day did."""
    # TODO: go through the days in chronological order
    # TODO: keep a set of every fingerprint seen so far
    # TODO: for each day, count the fingerprints not in that set, then add them to it
    raise NotImplementedError("your turn")


import pandas as pd

novelty = new_per_day(sessions)
print(f"NEW fingerprints per day:    median {int(novelty.median()):>5}, max {novelty.max():>5}")
print(f"days that brought nothing new: {(novelty == 0).sum()} of {len(novelty)}")

Stop and read those two numbers against each other.

A median day brings 135 sessions and **2** fingerprints the analyst has not
already characterised. The busiest day of the summer brought 12. Across 66 days
the entire collection produced 183 distinct intents - a list a person can read in
an afternoon, against 9 999 strings nobody can.

That is the difference between the two lanes in the lecture: the queue did not get
smaller, it got *finite*.

In [ ]:
fig, (top, bottom) = plt.subplots(2, 1, figsize=(8, 4.8), sharex=True)

top.fill_between(volume.index, volume.values, color=labs_lib.SERIES[0], alpha=0.25)
top.plot(volume.index, volume.values, color=labs_lib.SERIES[0])
top.set(ylabel="sessions", title="What arrives, and what is actually new")

bottom.bar(novelty.index, novelty.values, color=labs_lib.SERIES[1], width=1.0)
bottom.set(ylabel="new fingerprints", xlabel="")
bottom.set_ylim(0, max(novelty.max() * 1.3, 1))

for ax in (top, bottom):
    ax.margins(x=0.01)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

Two panels, not two y-axes on one chart: the scales differ by two orders of
magnitude and overlaying them would invent a relationship that is not in the data.

Now watch the big families move. Each of the top fingerprints gets its own panel,
on a shared time axis:

In [ ]:
top_four = counts.head(4).index

fig, axes = plt.subplots(len(top_four), 1, figsize=(8, 6.4), sharex=True)
for ax, fp, colour in zip(axes, top_four, labs_lib.SERIES):
    series = sessions[sessions.fingerprint == fp].groupby(sessions.date.dt.date).size()
    series = series.reindex(volume.index, fill_value=0)
    ax.fill_between(series.index, series.values, color=colour, alpha=0.3)
    ax.plot(series.index, series.values, color=colour)
    label = fp if len(fp) <= 58 else fp[:58] + "..."
    ax.set_title(f"{label}   ({counts[fp]:,} sessions)", fontsize=9, pad=4)
    ax.margins(x=0.01)

axes[0].figure.suptitle("One panel per intent, one shared timeline", x=0.02, ha="left",
                        fontweight="bold", fontsize=11)
fig.autofmt_xdate()
plt.tight_layout(rect=(0, 0, 1, 0.97))
plt.show()

> **Question 2.** None of these four panels looks like a campaign - and they look suspiciously alike.
>
> Print one session from each: are they really four different intents? 
>
> Then check which sensors they come from. What would you put in a ticket: a campaign, or something about your own honeypot?

## Part 3 - Which commands are ambiguous?

In Lab 1 you watched `rm` change tactic when its neighbours changed. That was one
command in three hand-picked sessions. With ten thousand labelled sessions you can
ask it of *every* command at once.

For each word, look at the distribution of tactics it receives across all its
occurrences, and measure how spread out that distribution is. Zero bits means the
word always means the same thing; high entropy means it is genuinely ambiguous and
only context resolves it.

In [ ]:
SEPARATORS = {";", "|", "||", "&&"}          # punctuation, not commands: they take the
                                             # tactic of whatever they sit between, so they
                                             # would top the ambiguity ranking for free
pairs = pd.DataFrame(
    [(w, t) for words, tactics in zip(sessions.sequence_words, sessions.sequence_predictions)
     for w, t in zip(words, tactics) if w not in SEPARATORS],
    columns=["word", "tactic"],
)
print(f"{len(pairs):,} labelled words, {pairs.word.nunique():,} distinct")
pairs.head(5)

In [ ]:
def label_entropy(pairs, min_count=200):
    """Shannon entropy, in bits, of each word's tactic distribution."""
    # TODO: cross-tabulate (`pd.crosstab`) words against tactics into a count matrix
    # TODO: keep only words seen at least `min_count` times - rare words are noise
    # TODO: turn each row into a probability distribution (frequency divided by row sum)
    # TODO: return -sum(p * log2 p) per word, sorted ascending, clipped at 0
    raise NotImplementedError("your turn")


entropy = label_entropy(pairs)
print(f"{len(entropy)} words seen at least 200 times\n")
print("always the same intent:", ", ".join(entropy.head(6).index))
print("depends on context:    ", ", ".join(entropy.tail(6).index))

In [ ]:
unambiguous = (entropy == 0).sum()
print(f"{unambiguous} of {len(entropy)} words carry exactly one tactic, every time they appear")
print("e.g.", ", ".join(entropy.head(8).index))

In [ ]:
ambiguous = entropy.tail(12)

fig, ax = plt.subplots(figsize=(7, 4.6))
ax.barh(range(len(ambiguous)), ambiguous.values, color=labs_lib.SERIES[1])
ax.set_yticks(range(len(ambiguous)), ambiguous.index, fontsize=9)
ax.set(xlabel="entropy of the tactic distribution (bits)", xlim=(0, ambiguous.max() * 1.15),
       title="The commands that need their neighbours")
ax.grid(axis="y", visible=False)
for i, value in enumerate(ambiguous.values):
    ax.text(value + 0.02, i, f"{value:.2f}", va="center", fontsize=8, color=labs_lib.INK_MUTED)
plt.tight_layout()
plt.show()

Read the top of that chart. `-rf` is the most context-dependent token in ten
thousand attacks - and it is exactly the one you watched flip from Defense Evasion
to Discovery in Lab 1 when a single `cat` was added after it.

That is the lecture's argument, arrived at from the other direction. A
non-contextual embedding has one vector for `-rf`, and the chart says that vector
would have to carry more than a bit of unresolved ambiguity. Attention is what
buys the difference.

> **Question 3.** Many words sit at exactly 0 bits (in practice: every time the
> word appears, the model gives it the same tactic). Look at how many sessions each
> of them appears in: what do you notice? Are they really unambiguous words, or just
> the same script repeated thousands of times?
>
> **Question 4.** These tactics come from the model, not from a human. Does a high
> entropy for `-rf` mean the word is truly ambiguous, or just that the model is
> unsure? How could you check with the labelled sessions from Lab 1?

## Part 4 - Your turn: identify a campaign

Pick one fingerprint and characterise it the way an analyst would. The scaffolding
below gets you the sessions; the analysis is yours.

The one selected is the sixth largest. Look at what the attacker actually typed
before you do anything else - it will not look like the others.

In [ ]:
chosen = counts.index[5]          # change this
members = sessions[sessions.fingerprint == chosen]

print(f"{chosen}\n{len(members):,} sessions | {members.sensor.nunique()} sensors "
      f"| {members.date.min():%b %d} to {members.date.max():%b %d}\n")
for text in members.session.head(2):
    print(text[:200], "\n")

The payload is **base64**. LogPrécis labelled it `Execution` without decoding
anything - the tactic is legible from the shape of the session alone.

You are the analyst, though, and you can decode it. Recover the indicators of
compromise hiding in there: the URLs the payload fetches.

In [ ]:
import base64
import collections
import re


def artefacts(members):
    """Every URL reachable in this family, decoding base64 payloads on the way."""
    # TODO: walk every word of every session in `members`
    # TODO: a long word is probably a base64 blob - strip quotes, pad with "==",
    #       and try to decode it; skip the ones that are not valid base64
    # TODO: collect the URLs from what you decode with a regex
    # TODO: return a Counter, so the ones that repeat rise to the top
    raise NotImplementedError("your turn")


for url, n in artefacts(members).most_common(5):
    print(f"{n:>5}  {url}")

There it is: one drop server, one archive, behind almost five hundred sessions -
no two of them identical. 

(The count is twice the sessions: every payload names the
URL twice, once for `wget` and once for `curl`.)

Decode one payload in full and read it:

In [ ]:
blob = next(w for w in members.iloc[0].sequence_words if len(w) > 60)
print(base64.b64decode(blob.strip('"') + "==").decode("utf8", "replace")[:520])

`dota2.tar.gz`. This is **DOTA**, also known as
[Outlaw](https://www.elastic.co/security-labs/outlaw-linux-malware) - the SSH
cryptomining worm the lecture used as its example, the one with >30 000 unique
sessions that all run the same tactics in the same order.

You did not go looking for it. You sorted ten thousand sessions by fingerprint,
opened the sixth group, and it was there - which is exactly the workflow the paper
is arguing for.

In [ ]:
print(f"sensors: {members.sensor.nunique()} of {sessions.sensor.nunique()}")
print(f"active:  {members.date.min():%b %d} to {members.date.max():%b %d}, "
      f"busiest {members.groupby(members.date.dt.date).size().idxmax()}")
print(f"distinct session strings: {members.session.nunique():,} of {len(members):,}")

> **Question 5.** Try `chosen = counts.index[0]` - the fingerprint covering 36% of
> all traffic. Run the same analysis. Why is it so much less interesting, and what
> does that say about ranking an analyst's queue by volume?

## Take-home

- You used one chunk of the collection. `labs_lib.FILES["sessions"]` points at
  `cyberlab_chunk_aa`; there are 24 more, and the published predictions cover all
  233 035 sessions. Redo Part 1 on five chunks. Does the compression ratio hold,
  or was it an artefact of a quiet summer on twenty sensors?
- Fingerprints are exact strings, so `Discovery x 54 -- Defense Evasion x 8` and
  `Discovery x 56 -- Defense Evasion x 6` are entirely different things - though
  they are plainly the same botnet. Drop the counts, or cluster by edit distance
  over tactic sequences, and see how much of the long tail collapses into the head.
- Lab 1 gave you a model that runs in milliseconds and a harness that was wrong
  until you fixed it. This one gave you a queue an analyst can actually work.
  Neither knew what a CVE is. **This afternoon**: an agent that does, and what it
  costs to ask it.